In [29]:
## IMPORTS AND DATASET LOADING
import numpy as np
import pandas as pd
import os

path = r"C:\Users\Pratham Shah\Downloads\Document dataset"
os.chdir(path)

df = pd.read_csv('monoculture_dataset.csv')


import xgboost as xgb
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

print(df.head())

    strain_id strain_type preferred_nutrient  maintenance_cost  growth_rate  \
0  Strain_001  specialist                 c1            0.5129       0.9993   
1  Strain_002  specialist                 c2            0.5750       1.1703   
2  Strain_003  specialist                 c3            0.5609       1.0491   
3  Strain_004  specialist                 c4            0.2082       1.0057   
4  Strain_005  specialist                 c5            0.5167       1.0844   

   delta dominant_nutrient  gene_1  gene_2  gene_3  ...  GO_00991  GO_00992  \
0  0.136               c17       0       0       1  ...         0         0   
1  0.136               c17       1       0       1  ...         0         0   
2  0.136               c17       1       1       0  ...         0         0   
3  0.136               c17       0       1       1  ...         0         1   
4  0.136               c17       0       1       1  ...         0         0   

   GO_00993  GO_00994  GO_00995  GO_00996  GO_0099

In [32]:
# 1. Keep the labels in a separate variable for later use in GNN
metadata = df[['strain_id','strain_type','preferred_nutrient','delta','dominant_nutrient']].copy()

drop_cols = ['strain_id','strain_type','preferred_nutrient','delta','dominant_nutrient','ss_suppression_score','ss_abundance','ss_luminescence']

X = df.drop(columns=drop_cols).copy()
y = df['ss_suppression_score'].copy()
 

 
print("Feature matrix shape:", X.shape)
print(X.head())

Feature matrix shape: (100, 1050)
   maintenance_cost  growth_rate  gene_1  gene_2  gene_3  gene_4  gene_5  \
0            0.5129       0.9993       0       0       1       1       1   
1            0.5750       1.1703       1       0       1       0       1   
2            0.5609       1.0491       1       1       0       1       1   
3            0.2082       1.0057       0       1       1       0       1   
4            0.5167       1.0844       0       1       1       0       0   

   gene_6  gene_7  gene_8  ...  GO_00991  GO_00992  GO_00993  GO_00994  \
0       1       1       1  ...         0         0         0         0   
1       0       0       1  ...         0         0         0         0   
2       0       0       0  ...         0         0         0         0   
3       1       0       1  ...         0         1         0         0   
4       1       0       1  ...         0         0         0         0   

   GO_00995  GO_00996  GO_00997  GO_00998  GO_00999  GO_01000  


In [33]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
## HYPERPARAMETER TUNING
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'gamma': [0, 0.25],
    'reg_lambda': [1, 5],
    'n_estimators': [100,300]
}

grid_search = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', seed=42),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)


Fitting 3 folds for each of 72 candidates, totalling 216 fits
Best Parameters: {'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'reg_lambda': 5}


In [35]:
## HYPERPARAMETER TUNING
param_grid2 = {
    'max_depth': [2,3,4],
    'learning_rate': [0.05],
    'gamma': [0],
    'reg_lambda': [5],
    'n_estimators': [300,400,500]
}

grid_search2 = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', seed=42),
    param_grid=param_grid2,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search2.fit(X_train, y_train)
print("Best Parameters:", grid_search2.best_params_)


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best Parameters: {'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 500, 'reg_lambda': 5}


In [36]:
## HYPERPARAMETER TUNING
param_grid3 = {
    'max_depth': [3],
    'learning_rate': [0.05],
    'gamma': [0],
    'reg_lambda': [5],
    'n_estimators': [600,700,800,900,1000]
}

grid_search3 = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', seed=42),
    param_grid=param_grid3,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search3.fit(X_train, y_train)
print("Best Parameters:", grid_search3.best_params_)


Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best Parameters: {'gamma': 0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 800, 'reg_lambda': 5}


In [37]:
# FINAL MODEL

reg_xgb_final = xgb.XGBRegressor(
    objective='reg:squarederror',
    seed=42,
    **grid_search3.best_params_,
    early_stopping_rounds=30,
    eval_metric='rmse'
)

reg_xgb_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Print the best tree number (iteration)
print(f"Best Iteration: {reg_xgb_final.best_iteration}")

# Print the best score achieved
print(f"Best Score: {reg_xgb_final.best_score}")


Best Iteration: 117
Best Score: 0.22820146762066446


In [39]:
predictions = reg_xgb_final.predict(X_test)
print(f"R2 Score: {r2_score(y_test, predictions):.4f}") 
print(f"Mean Absolute Error: {np.mean(np.abs(y_test - predictions)):.4f}")

print(predictions)

# Create a clean dataframe comparing actuals and predictions
comparison_df = pd.DataFrame({
    'Actual_Score': y_test.values,
    'Predicted_Score': predictions
})

# Round everything to 4 decimal places and show the first 20 rows
print(comparison_df.round(4).head(20))




R2 Score: 0.6572
Mean Absolute Error: 0.1379
[ 7.05330253e-01  8.41986947e-03  1.72116026e-01  7.83178210e-01
  9.04524267e-01  2.42021680e-02  1.37984693e-01  7.42431521e-01
  9.61775035e-02  7.62409389e-01  8.35943967e-04 -1.98706402e-03
 -1.69728533e-03  8.79280493e-02  7.72355974e-01  1.59270130e-02
  6.65939152e-02  1.96155861e-01  1.43467765e-02  4.90516499e-02]
    Actual_Score  Predicted_Score
0         1.0000           0.7053
1         0.0009           0.0084
2         0.0748           0.1721
3         0.6210           0.7832
4         0.9999           0.9045
5         0.0000           0.0242
6         0.0000           0.1380
7         1.0000           0.7424
8         0.0000           0.0962
9         0.0774           0.7624
10        0.0000           0.0008
11        0.0000          -0.0020
12        0.0000          -0.0017
13        0.0000           0.0879
14        0.7814           0.7724
15        0.0000           0.0159
16        0.6211           0.0666
17        0.0314 